# DoubleDeflector Before Two Lenses: Differentiable Beam Shift Optimization

This notebook demonstrates a differentiable `DoubleDeflector` placed before two lenses.
We optimize `drive_x`, `drive_y`, `balance_x`, `balance_y` to hit a target sample-plane shift while keeping output angle near zero.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import optimistix as optx

from temgym_core.ray import Ray
from temgym_core.components import DoubleDeflector, Deflector, Lens, Detector
from temgym_core.run import run_to_end
from temgym_core.plotting import plot_model, legacy_beam_plot_params

jax.config.update('jax_enable_x64', True)

## Geometry and Helper Builders

In [ ]:
z0 = 0.0
z_def = 0.08
spacing = 0.02
z_lens1 = 0.24
z_lens2 = 0.40
z_sample = 0.56

f1 = 0.20
f2 = 0.23

ray0 = Ray(x=0.0, y=0.0, dx=0.0, dy=0.0, z=z0, pathlength=0.0)

def build_model_double(params):
    drive_x, drive_y, balance_x, balance_y = params
    return (
        DoubleDeflector(
            z=z_def,
            spacing=spacing,
            drive_x=drive_x,
            drive_y=drive_y,
            balance_x=balance_x,
            balance_y=balance_y,
        ),
        Lens(z=z_lens1, focal_length=f1),
        Lens(z=z_lens2, focal_length=f2),
        Detector(z=z_sample, pixel_size=(1e-6, 1e-6), shape=(64, 64)),
    )


def run_sample(params, model_builder):
    return run_to_end(ray0, model_builder(params))

## Differentiable Optimization: Target Position with Near-Zero Angle

In [ ]:
target_xy = jnp.array([3.0e-4, -2.0e-4], dtype=jnp.float64)

def residual_physical(params):
    out = run_sample(params, build_model_double)
    return jnp.array([
        (out.x - target_xy[0]) / 1e-4,
        (out.y - target_xy[1]) / 1e-4,
        out.dx / 5e-5,
        out.dy / 5e-5,
    ], dtype=jnp.float64)


def bounded_logistic_to_physical(u, bounds_arr):
    b = jnp.asarray(bounds_arr, dtype=jnp.float64)
    lo = b[:, 0]
    hi = b[:, 1]
    return lo + (hi - lo) * jax.nn.sigmoid(u)


def bounded_logit_from_physical(x, bounds_arr):
    b = np.asarray(bounds_arr, dtype=float)
    lo = b[:, 0]
    hi = b[:, 1]
    s = np.clip((np.asarray(x, dtype=float) - lo) / (hi - lo), 1e-9, 1.0 - 1e-9)
    return np.log(s) - np.log1p(-s)


@jax.jit
def residual_unbounded_u(u, args):
    bounds_arr, = args
    x = bounded_logistic_to_physical(u, bounds_arr)
    return residual_physical(x)


x0 = np.array([1.0e-4, -1.0e-4, 1.0, 1.0], dtype=float)

bounds_arr = np.array([
    [-1e-2, 1e-2],
    [-1e-2, 1e-2],
    [0.2, 3.0],
    [0.2, 3.0],
], dtype=float)

x0 = np.clip(x0, bounds_arr[:, 0], bounds_arr[:, 1])
u0 = bounded_logit_from_physical(x0, bounds_arr)

r0 = np.asarray(residual_physical(jnp.asarray(x0, dtype=jnp.float64)))
loss_init = float(r0 @ r0)

solver = optx.LevenbergMarquardt(rtol=1e-12, atol=1e-12)
sol = optx.root_find(
    residual_unbounded_u,
    solver,
    y0=jnp.asarray(u0, dtype=jnp.float64),
    args=(bounds_arr,),
    options={'jac': 'fwd'},
    max_steps=800,
    throw=False,
)

u_star = np.asarray(sol.value, dtype=float)
x_star = np.asarray(bounded_logistic_to_physical(u_star, bounds_arr), dtype=float)

params_opt = jnp.asarray(x_star, dtype=jnp.float64)
rf = np.asarray(residual_physical(params_opt))
loss_final = float(rf @ rf)

loss_history = [loss_init, loss_final]

print('Solver result:', str(sol.result))
print('Num steps:', int(np.asarray(sol.stats.get('num_steps', 0))))
print(f'Initial loss: {loss_init:.6f}')
print(f'Final loss:   {loss_final:.6f}')
print('Optimized params [drive_x, drive_y, balance_x, balance_y]:')
print(np.asarray(params_opt))


In [ ]:
def pivot_distance(spacing_val, balance_val, eps=1e-12):
    if abs(float(balance_val) - 1.0) < eps:
        return np.inf
    return float(spacing_val / (float(balance_val) - 1.0))

out_opt = run_sample(params_opt, build_model_double)

d_pivot_x = pivot_distance(spacing, params_opt[2])
d_pivot_y = pivot_distance(spacing, params_opt[3])

print(f'Final sample x,y:  ({float(out_opt.x):+.6e}, {float(out_opt.y):+.6e}) m')
print(f'Target sample x,y: ({float(target_xy[0]):+.6e}, {float(target_xy[1]):+.6e}) m')
print(f'Final sample dx,dy: ({float(out_opt.dx):+.6e}, {float(out_opt.dy):+.6e}) rad')
print(f'Inferred d_pivot_x = {d_pivot_x:+.6e} m')
print(f'Inferred d_pivot_y = {d_pivot_y:+.6e} m')

## Tilt Invariance Check Over Many Drives

With fixed optimized balances, we sweep many drive values and verify sample-plane tilt remains ~0.

In [ ]:
def run_with_fixed_balance(drive_x, drive_y):
    params = jnp.array([
        drive_x,
        drive_y,
        params_opt[2],
        params_opt[3],
    ], dtype=jnp.float64)
    return run_sample(params, build_model_double)

# Axis-wise sweeps
drive_vals = np.linspace(-8e-3, 8e-3, 81)
dx_from_xdrive = []
dy_from_ydrive = []

for d in drive_vals:
    out_x = run_with_fixed_balance(float(d), 0.0)
    dx_from_xdrive.append(float(out_x.dx))

    out_y = run_with_fixed_balance(0.0, float(d))
    dy_from_ydrive.append(float(out_y.dy))

dx_from_xdrive = np.asarray(dx_from_xdrive)
dy_from_ydrive = np.asarray(dy_from_ydrive)

# Random combined drives
rng = np.random.default_rng(0)
combo = rng.uniform(-8e-3, 8e-3, size=(200, 2))
combo_tilts = []
for dx_d, dy_d in combo:
    out = run_with_fixed_balance(float(dx_d), float(dy_d))
    combo_tilts.append([float(out.dx), float(out.dy)])
combo_tilts = np.asarray(combo_tilts)

max_abs_dx_axis = np.max(np.abs(dx_from_xdrive))
max_abs_dy_axis = np.max(np.abs(dy_from_ydrive))
max_abs_dx_combo = np.max(np.abs(combo_tilts[:, 0]))
max_abs_dy_combo = np.max(np.abs(combo_tilts[:, 1]))

print(f'Max |dx| (x-drive sweep): {max_abs_dx_axis:.3e} rad')
print(f'Max |dy| (y-drive sweep): {max_abs_dy_axis:.3e} rad')
print(f'Max |dx| (random combos): {max_abs_dx_combo:.3e} rad')
print(f'Max |dy| (random combos): {max_abs_dy_combo:.3e} rad')

tilt_tol = 1e-12
assert max_abs_dx_axis < tilt_tol
assert max_abs_dy_axis < tilt_tol
assert max_abs_dx_combo < tilt_tol
assert max_abs_dy_combo < tilt_tol
print(f'Passed: tilt remains below {tilt_tol:.1e} rad for tested drive range.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(loss_history)
axes[0].set_title('Optimization Loss')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(float(out_opt.x), float(out_opt.y), label='Optimized', s=60)
axes[1].scatter(float(target_xy[0]), float(target_xy[1]), label='Target', s=60, marker='x')
axes[1].set_title('Sample-Plane Shift')
axes[1].set_xlabel('x (m)')
axes[1].set_ylabel('y (m)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()

In [ ]:
components_opt = build_model_double(params_opt)
plot_model(components_opt, rays=ray0)

## GIF: Side-View Beam Motion (Back and Forth)

This animation keeps optimized balances fixed, sweeps the drive scale forward and backward, and shows the beam path in side view (`x` vs `z`).

In [ ]:
from dataclasses import replace
from pathlib import Path
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from temgym_core.run import run_iter_vmapped

plt.rcParams['animation.html'] = 'none'

base_drive_x = float(params_opt[0])
base_drive_y = float(params_opt[1])
balance_x = float(params_opt[2])
balance_y = float(params_opt[3])

# Exactly 25 rays for bundle visualization in side view
bundle_radius = 2.0e-5
x_bundle = np.linspace(-bundle_radius, bundle_radius, 25)
rays_bundle = Ray(
    x=x_bundle,
    y=np.zeros_like(x_bundle),
    dx=np.zeros_like(x_bundle),
    dy=np.zeros_like(x_bundle),
    z=np.zeros_like(x_bundle) + float(ray0.z),
    pathlength=np.zeros_like(x_bundle),
)

# Forward then backward sweep
scales_fwd = np.linspace(-1.2, 1.2, 61)
scales = np.concatenate([scales_fwd, scales_fwd[-2:0:-1]])


def model_for_scale(s):
    params_s = jnp.array([
        s * base_drive_x,
        s * base_drive_y,
        balance_x,
        balance_y,
    ], dtype=jnp.float64)
    return build_model_double(params_s)


plot_style = legacy_beam_plot_params(
    fill_color='#7BFFF0',
    fill_alpha=0.55,
    solid_beam=True,
)


def detector_half_width_x(components):
    det_rx = 0.0
    for c in components:
        if isinstance(c, Detector):
            try:
                det_rx = max(det_rx, float(c.pixel_size[1] * c.shape[1] / 2.0))
            except Exception:
                det_rx = max(det_rx, float(c.pixel_size[0] * c.shape[0] / 2.0))
    return det_rx


def bundle_abs_x_max(components):
    steps = (rays_bundle,) + tuple(run_iter_vmapped(rays_bundle, components))
    return max(float(np.max(np.abs(np.asarray(r.x)))) for r in steps)


global_beam_x = 0.0
for s in scales:
    global_beam_x = max(global_beam_x, bundle_abs_x_max(model_for_scale(s)))

global_detector_x = detector_half_width_x(model_for_scale(scales[0]))
fixed_xmax = max(global_beam_x, global_detector_x, float(np.finfo(float).eps))
plot_style_fixed = replace(plot_style, fixed_xmax=fixed_xmax)

with plt.ioff():
    # One probe frame is enough now that horizontal scaling is fixed.
    fig_probe, ax_probe = plt.subplots(figsize=(6.0, 6.0))
    plot_model(
        model_for_scale(scales[0]),
        rays=rays_bundle,
        plot_params=plot_style_fixed,
        band_mode='fill',
        ax=ax_probe,
    )
    fixed_xlim = ax_probe.get_xlim()
    fixed_ylim = ax_probe.get_ylim()
    plt.close(fig_probe)

    fig_anim, ax_anim = plt.subplots(figsize=(6.0, 6.0))

    def _update(frame_idx):
        ax_anim.clear()
        s = scales[frame_idx]
        plot_model(
            model_for_scale(s),
            rays=rays_bundle,
            plot_params=plot_style_fixed,
            band_mode='fill',
            ax=ax_anim,
        )
        ax_anim.set_xlim(fixed_xlim)
        ax_anim.set_ylim(fixed_ylim)
        ax_anim.set_title(f'Side View (25-ray bundle), drive scale = {s:+.2f}')
        return []

    anim = FuncAnimation(fig_anim, _update, frames=len(scales), interval=70, blit=False)

    if Path('examples/ray_transfer/double_deflector_two_lens_shift_optimization.ipynb').exists():
        out_dir = Path('examples/ray_transfer')
    elif Path('double_deflector_two_lens_shift_optimization.ipynb').exists():
        out_dir = Path('.')
    else:
        out_dir = Path('.')
    out_dir.mkdir(parents=True, exist_ok=True)
    gif_path = out_dir / 'double_deflector_sideview_plot_model_25rays.gif'

    anim.save(gif_path, writer=PillowWriter(fps=15))
    plt.close(fig_anim)

display(Image(filename=str(gif_path)))
